# Assignment Case C — Customer RFM Analysis
## Working with PySpark DataFrames for Analytics
**Day 42 | Batch 13 | 03 Mei 2026**

---

| Field | Isi |
|-------|-----|
| **Nama** | *(isi nama kamu)* |
| **NIM** | *(isi NIM kamu)* |
| **Case** | Case C — Customer RFM Segmentation |
| **Source** | Hive: `adventureworks.*` |
| **Target** | Hive: `adventureworks_curated.fact_customer_rfm` |

---

## 🎯 Business Question

1. **Siapa customer paling berharga** (Champion & Loyal)?
2. **Customer mana yang perlu di-winback** (At Risk & Lost)?
3. **Distribusi segmen** — berapa % customer di tiap segmen?
4. **Rata-rata revenue** per segmen RFM?

## 📐 RFM Framework

```
R — Recency   : Berapa hari sejak order terakhir? (lebih kecil = lebih baik)
F — Frequency : Berapa kali order? (lebih besar = lebih baik)  
M — Monetary  : Berapa total spending? (lebih besar = lebih baik)
```

## 📊 Scoring 1–4 (Quartile-based)

| Dimensi | Score 4 (Terbaik) | Score 1 (Terburuk) |
|---------|-------------------|--------------------|
| **R** | Recency terkecil (baru beli) | Recency terbesar (lama tidak beli) |
| **F** | Frequency terbesar | Frequency terkecil |
| **M** | Monetary terbesar | Monetary terkecil |

> ⚠️ Recency di-score **terbalik**: recency kecil → score R tinggi

## 🏷️ Segmentasi

| Segment | Kondisi |
|---------|--------|
| **Champion** | R ≥ 3 AND F ≥ 3 AND M ≥ 3 |
| **Loyal** | F ≥ 3 AND M ≥ 3 (bukan Champion) |
| **Potential Loyal** | R ≥ 3 AND F >= 2 |
| **At Risk** | R <= 2 AND F >= 3 AND M >= 3 |
| **Lost** | R = 1 AND F <= 2 |
| **New Customer** | R = 4 AND F = 1 |
| **Others** | Semua yang tidak masuk kategori di atas |

## 📋 Kolom Target: `fact_customer_rfm`

| Kolom | Tipe | Keterangan |
|-------|------|------------|
| `customer_id` | INT | ID customer |
| `customer_name` | STRING | Nama customer |
| `territory_name` | STRING | Wilayah customer |
| `last_order_date` | DATE | Tanggal order terakhir |
| `recency_days` | INT | Hari sejak order terakhir |
| `frequency` | INT | Jumlah order |
| `monetary` | DECIMAL | Total spending |
| `r_score` | INT | Recency score (1-4) |
| `f_score` | INT | Frequency score (1-4) |
| `m_score` | INT | Monetary score (1-4) |
| `rfm_score` | STRING | Gabungan R+F+M score, contoh: '344' |
| `rfm_segment` | STRING | Champion/Loyal/At Risk/dll |
| `load_timestamp` | TIMESTAMP | Waktu load |


## 0. Setup SparkSession

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('CaseC_CustomerRFM') \
    .config('hive.metastore.uris', 'thrift://hive-metastore:9083') \
    .config('spark.sql.warehouse.dir', 'hdfs://namenode:9000/user/hive/warehouse') \
    .enableHiveSupport() \
    .getOrCreate()

# Reference date — pakai tanggal max order sebagai "hari ini"
# (bukan tanggal sekarang, karena data AdventureWorks historis)
REFERENCE_DATE = '2014-07-31'  # tanggal terakhir di dataset AdventureWorks

print(f'Spark {spark.version} ready')
print(f'Reference date: {REFERENCE_DATE}')

## 1. Load Data dari Hive

In [ ]:
# TODO: Load tabel yang dibutuhkan dari Hive
# Petunjuk: perlu fact_sales_orders (customerid, orderdate, totaldue)
#           dim_customer (customerid, personid, territoryid)
#           dim_person (businessentityid, firstname, lastname)
#           dim_territory (territoryid, name)

df_orders    = ___  # TODO
df_customer  = ___  # TODO
df_person    = ___  # TODO
df_territory = ___  # TODO

print('Row counts:')
for name, df in [('fact_sales_orders', df_orders), ('dim_customer', df_customer),
                  ('dim_person', df_person), ('dim_territory', df_territory)]:
    print(f'  {name}: {df.count():,}')

## 2. Eksplorasi Data

In [ ]:
# TODO: Eksplorasi
# - Rentang tanggal order (min & max orderdate)
# - Jumlah customer unik
# - Distribusi total order per customer

print('=== Rentang Tanggal Order ===')
# TODO

print('\n=== Jumlah Customer Unik ===')
# TODO

print('\n=== Distribusi Order Frequency per Customer ===')
# TODO: hitung frequency per customer, lalu describe()

## 3. Hitung R, F, M per Customer

**Tugas:** Dari `fact_sales_orders`, hitung Recency, Frequency, Monetary per customer.

In [ ]:
# TODO: Hitung RFM raw values
# Recency   = datediff(reference_date, max(orderdate))  ← hari sejak order terakhir
# Frequency = count(salesorderid)                       ← total order
# Monetary  = round(sum(totaldue), 2)                   ← total spending

df_rfm_raw = df_orders \
    .groupBy('customerid') \
    .agg(
        ___  # TODO: recency_days
        ___  # TODO: frequency
        ___  # TODO: monetary
        F.max('orderdate').alias('last_order_date')
    )

print(f'Total customers: {df_rfm_raw.count():,}')
df_rfm_raw.describe().show()
df_rfm_raw.orderBy(F.asc('recency_days')).show(10)

## 4. Scoring R, F, M (Quartile-based)

Gunakan `ntile(4)` window function untuk membagi customer menjadi 4 kuartil.

> **Ingat:** R-score **terbalik** — recency kecil = baru beli = score 4 (bagus).

In [ ]:
# TODO: Hitung R, F, M score menggunakan ntile(4)
# Petunjuk:
# - R score: ntile(4) OVER (ORDER BY recency_days DESC)  ← terbalik! desc = kecil dapat rank 1→4
#   tapi kita mau recency kecil = score 4, jadi pakai:
#   ntile(4) OVER (ORDER BY recency_days ASC) lalu (5 - ntile_value)
#   ATAU langsung: ntile(4) OVER (ORDER BY recency_days DESC)
# - F score: ntile(4) OVER (ORDER BY frequency ASC)
# - M score: ntile(4) OVER (ORDER BY monetary ASC)

win_r = Window.orderBy(F.desc('recency_days'))   # recency kecil → ntile 1 → score 4
win_f = Window.orderBy(F.asc('frequency'))        # frequency besar → ntile 4 → score 4
win_m = Window.orderBy(F.asc('monetary'))         # monetary besar → ntile 4 → score 4

df_scored = df_rfm_raw \
    .withColumn('r_raw',  F.ntile(4).over(win_r)) \
    .withColumn('f_score', F.ntile(4).over(win_f)) \
    .withColumn('m_score', F.ntile(4).over(win_m)) \
    .withColumn('r_score', F.lit(5) - F.col('r_raw'))  # invert: ntile 1 → score 4

# Tambahkan rfm_score sebagai string gabungan misal '344'
df_scored = df_scored.withColumn(
    'rfm_score',
    F.concat(F.col('r_score').cast('string'),
             F.col('f_score').cast('string'),
             F.col('m_score').cast('string'))
)

print('=== Score Distribution ===')
df_scored.groupBy('r_score', 'f_score', 'm_score').count().orderBy('r_score','f_score','m_score').show(20)
print(f'Total customers scored: {df_scored.count():,}')

## 5. Segmentasi Customer

In [ ]:
# TODO: Assign segment berdasarkan R, F, M score
# Gunakan F.when().when().otherwise()
#
# Champion      : R>=3 AND F>=3 AND M>=3
# Loyal         : F>=3 AND M>=3 (bukan Champion)
# Potential Loyal: R>=3 AND F>=2
# At Risk       : R<=2 AND F>=3 AND M>=3
# Lost          : R=1 AND F<=2
# New Customer  : R=4 AND F=1
# Others        : otherwise

r = F.col('r_score')
f = F.col('f_score')
m = F.col('m_score')

df_segmented = df_scored.withColumn(
    'rfm_segment',
    F.when((r >= 3) & (f >= 3) & (m >= 3), 'Champion')
     .when(___  # TODO: Loyal
     .when(___  # TODO: Potential Loyal
     .when(___  # TODO: At Risk
     .when(___  # TODO: Lost
     .when(___  # TODO: New Customer
     .otherwise('Others')
)

print('=== Distribusi Segmen ===')
df_segmented.groupBy('rfm_segment') \
    .agg(
        F.count('customerid').alias('jumlah_customer'),
        F.round(F.avg('monetary'), 2).alias('avg_spending'),
        F.round(F.avg('frequency'), 1).alias('avg_frequency')
    ) \
    .orderBy(F.desc('jumlah_customer')) \
    .show()

## 6. Enrich dengan Data Customer

In [ ]:
# TODO: Join dengan dim_customer, dim_person, dim_territory
# Hasilnya: satu baris per customer dengan nama, wilayah, dan semua RFM info

df_final_rfm = df_segmented \
    .join(df_customer.select('customerid', 'personid', 'territoryid'), 'customerid', 'left') \
    .join(
        df_person.select(
            F.col('businessentityid').alias('personid'),
            F.concat_ws(' ', 'firstname', 'lastname').alias('customer_name')
        ), 'personid', 'left') \
    .join(F.broadcast(
        df_territory.select(
            'territoryid',
            F.col('name').alias('territory_name')
        )), 'territoryid', 'left') \
    .select(
        'customerid', 'customer_name', 'territory_name',
        'last_order_date', 'recency_days', 'frequency', 'monetary',
        'r_score', 'f_score', 'm_score', 'rfm_score', 'rfm_segment'
    ) \
    .withColumn('load_timestamp', F.current_timestamp())

print(f'Final RFM rows: {df_final_rfm.count():,}')
df_final_rfm.show(10, truncate=False)

## 7. Load — Simpan ke Hive

In [ ]:
spark.sql('CREATE DATABASE IF NOT EXISTS adventureworks_curated')

# TODO: simpan ke adventureworks_curated.fact_customer_rfm
# mode: overwrite, saveAsTable

print('=== Verifikasi ===')
df_verify = spark.table('adventureworks_curated.fact_customer_rfm')
print(f'Total rows: {df_verify.count():,}')
df_verify.show(5)

## 8. Analytic Queries

**Wajib: minimal 3 query yang menjawab business question**

In [ ]:
# ── Query 1 (WAJIB): Top 20 Champion Customers ─────────────────────────────
# Tampilkan customer segment Champion, sort by monetary desc
# Kolom: customer_name, territory_name, recency_days, frequency, monetary, rfm_score

print('=== Query 1: Top 20 Champion Customers ===')
spark.sql("""
    -- TODO: query dari adventureworks_curated.fact_customer_rfm
    SELECT 1
""")


In [ ]:
# ── Query 2 (WAJIB): Distribusi Segmen + Revenue Kontribusi ────────────────
# Tampilkan: rfm_segment, jumlah_customer, pct_customer, total_revenue, pct_revenue
# pct = % dari total

print('=== Query 2: Distribusi Segmen & Revenue Contribution ===')
# TODO: Gunakan window function sum() OVER () untuk hitung total keseluruhan

In [ ]:
# ── Query 3 (WAJIB): At Risk & Lost Customers per Territory ────────────────
# Tampilkan pelanggan yang perlu di-winback
# Kolom: territory_name, rfm_segment, count, avg_monetary, avg_recency_days

print('=== Query 3: At Risk & Lost Customers per Territory ===')
# TODO

In [ ]:
# ── Query 4 (BONUS): Average RFM metrics per Segment ──────────────────────
# Tampilkan profil rata-rata tiap segmen
# Kolom: rfm_segment, avg_recency, avg_frequency, avg_monetary, count_customers
# Sort: avg_monetary desc

print('=== Query 4 (Bonus): Profil Rata-rata per Segmen ===')
# TODO

In [ ]:
# ── Query 5 (BONUS): Segmen Paling Banyak per Territory ───────────────────
# Untuk tiap territory, segmen mana yang paling dominan?
# Gunakan window function rank() partitionBy territory_name, orderBy desc count

print('=== Query 5 (Bonus): Dominant Segment per Territory ===')
# TODO

In [ ]:
spark.stop()
print('Done!')